# verl处理“训练”与“推理”底层引擎不同，导致的logprob 口径不一致

针对大模型强化学习（RL）中由于“训练”与“推理”底层引擎不同，导致的 logprob 口径不一致（即 Importance Ratio 计算失准）这一致命痛点，VeRL（Volcano Engine Reinforcement Learning）框架在设计上通过以下三个层面进行了系统性的处理和规避：

## 一、 核心策略：弃用推理侧概率，强制训练侧重算
<font color='red'>这是 VeRL 框架中最根本、最核心的解决方案</font>

在训练阶段，当拿到推理引擎（如 vLLM）生成的 rollout 序列后，<font color='red'>VeRL 绝对不会直接使用推理引擎生成时返回的 logprob 来作为分母</font>（即旧策略概率 $π_{old}$）计算 ratio。

- 处理方式：
  - VeRL 会在训练阶段（Training Stage）<font color='red'>将生成的文本喂回训练引擎</font>（如 FSDP / Megatron），重新前向传播（Recompute） 计算出一套标准化的、精确的 old_log_probs。
- 底层原因：
  - 推理引擎为了追求高吞吐，通常会走“快路径”（Fast Path），
    - 在底层计算 logits 时可能会引入温度（temperature）、top-p 截断、重复惩罚（repetition penalty）<font color='red'>等处理器改写 logit</font>，**甚至底层算子的累加顺序与训练侧不同**。
  - <font color='red'>推理侧返回的 logprob 是“被采样策略污染”的</font>，直接用于训练会导致梯度方向错误。**而训练侧的重算使用的是原始、干净的前向计算，能够保证分子和分母在同一套标准下比较。**

## 二、 兜底策略：Bypass 模式（快速但有风险的捷径）
VeRL 允许用户通过配置开启一种名为 Bypass 模式 的捷径。
- 处理方式：
  - 在这种模式下，系统会跳过训练侧耗时的 logprob 重算步骤，直接用推理引擎返回的 rollout_log_probs 充当 old_log_probs 来计算 PPO ratio。
- 风险提示：
  - <font color='red'>这本质上是一种用“数值精度”换“训练速度”的妥协</font>。
    - <font color='blue'>只有当你确认推理引擎和训练引擎在底层数值上做到了高度一致（例如 tokenizer 完全对齐、无特殊采样器干扰、序列长度适中且模型较小）时，才建议使用。</font>对于 DeepSeek-V3 等长 CoT（思维链）或训推并行度差异大的场景，极易导致 ratio 方差失控。

## 三、 容错策略：Rollout Correction（训推失配修正）
对于那些为了极致性能（如 Fully Async 全异步训练）不得不接受训推参数不同步或底层计算存在固有微小差异的场景，VeRL 提供了 Rollout Correction 机制。

- 处理方式：
  - 它将 PPO 的 Loss 计算进行了数学上的解耦。它并不强行要求 $π_{old}$ 绝对等于 rollout 时的行为策略概率，而是将 rollout 阶段的策略（ $π_{rollout}$）和 training 阶段的策略（ $π_{old}$）<font color='red'>分开处理</font>
- 底层逻辑：
  - PPO 传统的 Clip 约束负责限制“本轮 optimizer 更新步长”；
  - <font color='red'>而对于训推不一致带来的偏差，框架可以通过额外的 IS（重要性采样）加权、截断（truncation）或拒绝掩码（rejection mask）来进行修正补偿，从而允许一定程度的训推底层误差存在，保证训练流程的稳定性。</font>